En estadística y aprendizaje automático, AIC y BIC son dos criterios utilizados para evaluar modelos, especialmente en el contexto de modelos de regresión y selección de modelos. Estos criterios ayudan a determinar qué modelo tiene un mejor ajuste y complejidad, penalizando aquellos con demasiados parámetros (para evitar sobreajuste). A partir de esto, se pueden utilizar técnicas de selección de variables como *forward selection*, *backward elimination* y *stepwise selection* para escoger las variables más relevantes. A continuación, te explico cada uno de estos conceptos.

### 1. Criterios de Información de Akaike (AIC) y Bayesiano (BIC)

Ambos criterios son medidas de ajuste de modelos que incluyen una penalización por la cantidad de parámetros. Son especialmente útiles cuando estamos probando modelos de distinta complejidad y queremos comparar cuál se ajusta mejor a los datos.

#### **AIC (Akaike Information Criterion)**

- **Fórmula:**
  \[
  AIC = 2k - 2 \ln(L)
  \]
  donde:
  - \( k \) es el número de parámetros en el modelo,
  - \( L \) es la verosimilitud del modelo dado el conjunto de datos.

- **Interpretación:** El AIC busca el modelo que mejor se ajusta a los datos sin sobreajustar. Penaliza ligeramente más el ajuste que la complejidad. Entre dos modelos, el que tenga menor AIC es preferido.

#### **BIC (Bayesian Information Criterion)**

- **Fórmula:**
  \[
  BIC = k \ln(n) - 2 \ln(L)
  \]
  donde:
  - \( n \) es el tamaño de la muestra.

- **Interpretación:** El BIC penaliza la complejidad de forma más fuerte que el AIC, especialmente en muestras grandes. Este criterio tiende a seleccionar modelos más simples que el AIC, porque la penalización aumenta con el tamaño de los datos.

Ambos criterios ayudan a encontrar un equilibrio entre ajuste y simplicidad del modelo, y usualmente se usan como guías, no como reglas estrictas. Un modelo con un AIC o BIC más bajo es mejor, pero no siempre el más bajo es el más interpretativo o útil.

### 2. Métodos de Selección de Variables: Forward, Backward y Stepwise

Estos métodos buscan identificar el subconjunto óptimo de variables para incluir en un modelo. El objetivo es maximizar el ajuste sin sobreajustar, idealmente seleccionando un subconjunto pequeño de variables significativas.

#### **Método de Selección Hacia Adelante (Forward Selection)**

- Empieza con un modelo sin variables.
- Agrega iterativamente la variable que más mejora el criterio de ajuste (AIC o BIC), usualmente de mayor a menor mejora en el ajuste.
- Detiene la inclusión de variables cuando ya no hay una mejora significativa en el criterio.

Este método es útil cuando partimos de un conjunto amplio de variables y queremos identificar cuáles son las más relevantes para el modelo.

#### **Método de Eliminación Hacia Atrás (Backward Elimination)**

- Comienza con un modelo que incluye todas las variables.
- En cada paso, elimina la variable que menos contribuye al ajuste del modelo (la que menos mejora el AIC o BIC).
- Continua eliminando variables hasta que el modelo no pueda mejorar significativamente.

Este método es útil cuando partimos de un conjunto pequeño o moderado de variables y queremos simplificar el modelo.

#### **Método de Selección Paso a Paso (Stepwise Selection)**

- Es una combinación de los métodos hacia adelante y hacia atrás.
- En cada paso, permite tanto agregar como eliminar variables, dependiendo de si mejoran o empeoran el ajuste (basado en AIC o BIC).
- Esto permite más flexibilidad y, a menudo, da un mejor modelo final, ya que se consideran ambas direcciones de inclusión y exclusión.

### Resumen

| Método             | Inicia con | Procedimiento |
|--------------------|------------|---------------|
| **Forward**        | Modelo vacío | Agrega variables una por una, basado en mejora del ajuste |
| **Backward**       | Modelo completo | Elimina variables una por una, basándose en el ajuste |
| **Stepwise**       | Puede iniciar vacío o completo | Alterna entre agregar y quitar variables según el ajuste |

### Aplicación Práctica

1. **Elegir el criterio**: AIC o BIC, dependiendo de si prefieres un modelo más complejo (AIC) o uno más sencillo (BIC).
2. **Aplicar el método**: Según el número de variables y el enfoque (agregar o quitar), elige el método de selección.
3. **Evaluar el resultado**: Observa el AIC/BIC final y asegúrate de que el modelo se ajuste bien a los datos y sea interpretativamente útil.

Estos métodos son muy utilizados en modelado estadístico y aprendizaje automático, ya que permiten construir modelos eficientes sin sobreajustar, balanceando ajuste y simplicidad.

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from statsmodels.tools import add_constant
import statsmodels.api as sm

# Cargar el conjunto de datos California Housing
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Función para calcular AIC y BIC
def calculate_aic_bic(model, X, y):
    X_sm = add_constant(X)  # Agregar constante para el intercepto
    model_sm = sm.OLS(y, X_sm).fit()
    return model_sm.aic, model_sm.bic

# 1. Método de Selección Hacia Adelante (Forward Selection)
forward_selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=5, direction='forward')
forward_selector.fit(X_train, y_train)
selected_forward = X_train.columns[forward_selector.get_support()]
print("Variables seleccionadas por Forward Selection:", selected_forward)

# Calcular AIC y BIC para el modelo Forward
aic_forward, bic_forward = calculate_aic_bic(LinearRegression(), X_train[selected_forward], y_train)
print("AIC Forward:", aic_forward)
print("BIC Forward:", bic_forward)

# 2. Método de Eliminación Hacia Atrás (Backward Elimination)
backward_selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=5, direction='backward')
backward_selector.fit(X_train, y_train)
selected_backward = X_train.columns[backward_selector.get_support()]
print("Variables seleccionadas por Backward Elimination:", selected_backward)

# Calcular AIC y BIC para el modelo Backward
aic_backward, bic_backward = calculate_aic_bic(LinearRegression(), X_train[selected_backward], y_train)
print("AIC Backward:", aic_backward)
print("BIC Backward:", bic_backward)

# 3. Método de Selección Paso a Paso (Stepwise Selection)
def stepwise_selection(X, y, initial_list=[], threshold_in=0.01, threshold_out=0.05):
    included = list(initial_list)
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True

        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)

        if not changed:
            break

    return included

# Ejecutar Stepwise selection
selected_stepwise = stepwise_selection(X_train, y_train)
print("Variables seleccionadas por Stepwise Selection:", selected_stepwise)

# Calcular AIC y BIC para el modelo Stepwise
aic_stepwise, bic_stepwise = calculate_aic_bic(LinearRegression(), X_train[selected_stepwise], y_train)
print("AIC Stepwise:", aic_stepwise)
print("BIC Stepwise:", bic_stepwise)

# Comparación de Modelos
print("\nComparación de Modelos:")
print(f"Forward Selection - AIC: {aic_forward}, BIC: {bic_forward}")
print(f"Backward Elimination - AIC: {aic_backward}, BIC: {bic_backward}")
print(f"Stepwise Selection - AIC: {aic_stepwise}, BIC: {bic_stepwise}")

Variables seleccionadas por Forward Selection: Index(['MedInc', 'HouseAge', 'AveBedrms', 'Latitude', 'Longitude'], dtype='object')
AIC Forward: 36439.22847622834
BIC Forward: 36485.49953223803
Variables seleccionadas por Backward Elimination: Index(['MedInc', 'HouseAge', 'AveBedrms', 'Latitude', 'Longitude'], dtype='object')
AIC Backward: 36439.22847622834
BIC Backward: 36485.49953223803
Variables seleccionadas por Stepwise Selection: ['MedInc', 'HouseAge', 'Latitude', 'Longitude', 'AveBedrms', 'AveRooms', 'AveOccup']
AIC Stepwise: 36188.37407476429
BIC Stepwise: 36250.068816110535

Comparación de Modelos:
Forward Selection - AIC: 36439.22847622834, BIC: 36485.49953223803
Backward Elimination - AIC: 36439.22847622834, BIC: 36485.49953223803
Stepwise Selection - AIC: 36188.37407476429, BIC: 36250.068816110535


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from statsmodels.tools import add_constant
import statsmodels.api as sm
from sklearn.datasets import fetch_openml

# Cargar el conjunto de datos Ames Housing desde OpenML
data = fetch_openml(name="house_prices", as_frame=True)
df = data.frame

# Seleccionar características relevantes y variable objetivo
X = df.select_dtypes(include=[np.number]).drop(columns="SalePrice").fillna(0)
y = df["SalePrice"]

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Función para calcular AIC y BIC
def calculate_aic_bic(model, X, y):
    X_sm = add_constant(X)  # Agregar constante para el intercepto
    model_sm = sm.OLS(y, X_sm).fit()
    return model_sm.aic, model_sm.bic

# 1. Método de Selección Hacia Adelante (Forward Selection)
forward_selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=5, direction='forward')
forward_selector.fit(X_train, y_train)
selected_forward = X_train.columns[forward_selector.get_support()]
print("Variables seleccionadas por Forward Selection:", selected_forward)

# Calcular AIC y BIC para el modelo Forward
aic_forward, bic_forward = calculate_aic_bic(LinearRegression(), X_train[selected_forward], y_train)
print("AIC Forward:", aic_forward)
print("BIC Forward:", bic_forward)

# 2. Método de Eliminación Hacia Atrás (Backward Elimination)
backward_selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=5, direction='backward')
backward_selector.fit(X_train, y_train)
selected_backward = X_train.columns[backward_selector.get_support()]
print("Variables seleccionadas por Backward Elimination:", selected_backward)

# Calcular AIC y BIC para el modelo Backward
aic_backward, bic_backward = calculate_aic_bic(LinearRegression(), X_train[selected_backward], y_train)
print("AIC Backward:", aic_backward)
print("BIC Backward:", bic_backward)

# 3. Método de Selección Paso a Paso (Stepwise Selection)
def stepwise_selection(X, y, initial_list=[], threshold_in=0.01, threshold_out=0.05):
    included = list(initial_list)
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True

        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)

        if not changed:
            break

    return included

# Ejecutar Stepwise selection
selected_stepwise = stepwise_selection(X_train, y_train)
print("Variables seleccionadas por Stepwise Selection:", selected_stepwise)

# Calcular AIC y BIC para el modelo Stepwise
aic_stepwise, bic_stepwise = calculate_aic_bic(LinearRegression(), X_train[selected_stepwise], y_train)
print("AIC Stepwise:", aic_stepwise)
print("BIC Stepwise:", bic_stepwise)

# Comparación de Modelos
print("\nComparación de Modelos:")
print(f"Forward Selection - AIC: {aic_forward}, BIC: {bic_forward}")
print(f"Backward Elimination - AIC: {aic_backward}, BIC: {bic_backward}")
print(f"Stepwise Selection - AIC: {aic_stepwise}, BIC: {bic_stepwise}")

Variables seleccionadas por Forward Selection: Index(['OverallQual', 'BsmtFinSF1', 'TotalBsmtSF', 'GrLivArea', 'GarageArea'], dtype='object')
AIC Forward: 27738.565058197728
BIC Forward: 27768.943347178058
Variables seleccionadas por Backward Elimination: Index(['OverallQual', 'YearBuilt', 'BsmtUnfSF', 'TotalBsmtSF', 'GrLivArea'], dtype='object')
AIC Backward: 27748.349242911743
BIC Backward: 27778.727531892073
Variables seleccionadas por Stepwise Selection: ['OverallQual', 'GrLivArea', 'BsmtFinSF1', 'GarageArea', 'TotalBsmtSF', 'YearRemodAdd', 'MSSubClass', 'MasVnrArea', 'LotArea', 'BedroomAbvGr', 'YearBuilt', 'OverallCond', 'GarageYrBlt', 'TotRmsAbvGrd', 'KitchenAbvGr', 'Fireplaces', 'LotFrontage']
AIC Stepwise: 27476.489651575368
BIC Stepwise: 27567.624518516353

Comparación de Modelos:
Forward Selection - AIC: 27738.565058197728, BIC: 27768.943347178058
Backward Elimination - AIC: 27748.349242911743, BIC: 27778.727531892073
Stepwise Selection - AIC: 27476.489651575368, BIC: 27567.6